# A1 corregido: Regresión logística (corrección i)

Cambios respecto a `A1_RegresionLogistica.ipynb` (el original queda intacto):
1. La etiqueta es binaria, `was_preterm` (0 = término, 1 = pretérmino), igual que en A2 y A3.
   Antes eran 3 clases (`term`, `preterm`, `early_preterm`).
2. El `train_test_split` por muestra se reemplaza por los pliegues comunes por paciente
   (`pliegues_por_paciente.csv`, 5 pliegues x 5 repeticiones).
3. La probabilidad de cada paciente es el promedio de las de sus muestras, y se evalúa por paciente.

Lo que ve el modelo (solo la categoría racial codificada como entero) **no** cambia aquí: es del punto iv.

In [1]:
import os, sys
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

sys.path.insert(0, os.path.abspath('.'))
from comun import (DATOS, RESULTADOS, ETIQUETA, cargar_pliegues, iterar_pliegues,
                   promedio_por_paciente, umbral_sens_igual_espec, metricas)
RESULTADOS.mkdir(exist_ok=True)

data = pd.read_csv(DATOS / 'metadata.csv')

label_encoder = LabelEncoder()
feature_race = label_encoder.fit_transform(data['NIH Racial Category'].fillna("Unknown")).reshape(-1, 1)
target = data[ETIQUETA].astype(int).values          # CAMBIO: binaria was_preterm
pacientes = data['participant_id'].values
print(data.shape, '| muestras pretérmino:', target.mean().round(3))

(1895, 12) | muestras pretérmino: 0.355


In [2]:
pliegues = cargar_pliegues()
filas, predicciones = [], []
for rep, fold, ids_tr, ids_te in iterar_pliegues(pliegues):
    tr, te = np.isin(pacientes, list(ids_tr)), np.isin(pacientes, list(ids_te))
    model = LogisticRegression(max_iter=200)
    model.fit(feature_race[tr], target[tr])
    prob_muestra = model.predict_proba(feature_race[te])[:, 1]
    pp = promedio_por_paciente(pacientes[te], target[te], prob_muestra)
    umbral = umbral_sens_igual_espec(pp.y, pp.prob)   # misma regla para los tres modelos (punto iii)
    filas.append(dict(modelo='Regresión logística', rep=rep, fold=fold, **metricas(pp.y, pp.prob, umbral)))
    predicciones.append(pp.assign(rep=rep, fold=fold))

pd.concat(predicciones).to_csv(RESULTADOS / 'pred_A1.csv', index=False)
res = pd.DataFrame(filas)
res.to_csv(RESULTADOS / 'metricas_A1.csv', index=False)
res[['auc_roc', 'auc_pr', 'accuracy', 'precision', 'recall', 'f1', 'umbral']].agg(['mean', 'std']).round(3)

,auc_roc,auc_pr,accuracy,precision,recall,f1,umbral
mean,0.547,0.480,0.530,0.468,0.602,0.526,0.383
std,0.040,0.031,0.036,0.033,0.041,0.031,0.013
